In [11]:
import polars as pl

df = pl.scan_parquet('../dataset/final_train_data/merged_train_data.parquet')


In [13]:
numeric_cols = df.select(pl.col(pl.Float32, pl.Float64, pl.Int32, pl.Int16, pl.Int8)).columns
numeric_cols = [c for c in numeric_cols if c not in ['customer_ID', 'target']]

cleaned_data = df.with_columns([
    pl.col(numeric_cols).fill_null(-999)
])


cleaned_data = cleaned_data.with_columns([
    pl.col("S_2").fill_null(pl.date(1900, 1, 1))
])

C:\Users\HP\AppData\Local\Temp\ipykernel_1748\415172586.py:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  numeric_cols = df.select(pl.col(pl.Float32, pl.Float64, pl.Int32, pl.Int16, pl.Int8)).columns


In [14]:
all_features = [c for c in cleaned_data.columns if c not in ["customer_ID","target","S_2"]]
volatility_exprs = [
    *[pl.col(c).mean().alias(f"{c}_mean") for c in all_features],
    *[pl.col(c).last().alias(f"{c}_last") for c in all_features],
    *[pl.col(c).std().alias(f"{c}_std") for c in all_features],  
    *[(pl.col(c).mean() - pl.col(c).last()).alias(f"{c}_delta") for c in all_features],
]


volatility_df = (
    cleaned_data.lazy()
    .group_by(["customer_ID","target"])
    .agg(volatility_exprs)
    .collect(streaming = True)
)

volatility_df.write_parquet("volatility_data.parquet")

C:\Users\HP\AppData\Local\Temp\ipykernel_1748\2123691541.py:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  all_features = [c for c in cleaned_data.columns if c not in ["customer_ID","target","S_2"]]
C:\Users\HP\AppData\Local\Temp\ipykernel_1748\2123691541.py:14: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming = True)


In [20]:
print(volatility_df.collect_schema())

Schema({'customer_ID': String, 'target': Int64, 'P_2_mean': Float32, 'D_39_mean': Float64, 'B_1_mean': Float32, 'B_2_mean': Float32, 'R_1_mean': Float32, 'S_3_mean': Float32, 'D_41_mean': Float32, 'B_3_mean': Float32, 'D_42_mean': Float32, 'D_43_mean': Float32, 'D_44_mean': Float64, 'B_4_mean': Float64, 'D_45_mean': Float32, 'B_5_mean': Float32, 'R_2_mean': Float64, 'D_46_mean': Float32, 'D_47_mean': Float32, 'D_48_mean': Float32, 'D_49_mean': Float64, 'B_6_mean': Float32, 'B_7_mean': Float32, 'B_8_mean': Float32, 'D_50_mean': Float32, 'D_51_mean': Float64, 'B_9_mean': Float32, 'R_3_mean': Float64, 'D_52_mean': Float32, 'P_3_mean': Float32, 'B_10_mean': Float32, 'D_53_mean': Float32, 'S_5_mean': Float32, 'B_11_mean': Float32, 'S_6_mean': Float64, 'D_54_mean': Float32, 'R_4_mean': Float64, 'S_7_mean': Float32, 'B_12_mean': Float32, 'S_8_mean': Float64, 'D_55_mean': Float32, 'D_56_mean': Float32, 'B_13_mean': Float32, 'R_5_mean': Float64, 'D_58_mean': Float32, 'S_9_mean': Float32, 'B_14_

In [17]:
from lightgbm import LGBMClassifier

In [19]:
import pandas as pd
all_importances = []
n_iterations = 3

defaults_df = volatility_df.filter(pl.col("target") == 1)
non_defaults_df = volatility_df.filter(pl.col("target") == 0)

for i in range(n_iterations):
    print(f"Iteration {i+1}/{n_iterations}...")
    
    sample = pl.concat([
        defaults_df.sample(n=100000, seed=i),
        non_defaults_df.sample(n=100000, seed=i)
    ])
    
   
    drop_cols = ["customer_ID", "target", "S_2"]
    X = sample.drop([c for c in drop_cols if c in sample.columns]).to_pandas()
    y = sample["target"].to_pandas()
    
    if i == 0:
        constant_cols = [col for col in X.columns if X[col].nunique() <= 1]
        print(f"Dropping {len(constant_cols)} constant columns.")
    
    X = X.drop(columns=constant_cols)
    
    selector = LGBMClassifier(
        n_estimators=250, 
        importance_type='gain', 
        random_state=i,
        device="gpu"
    )
    selector.fit(X, y)
    
    imp = pd.DataFrame({'feature': X.columns, f'gain_{i}': selector.feature_importances_})
    all_importances.append(imp.set_index('feature'))


final_rank = pd.concat(all_importances, axis=1).mean(axis=1).sort_values(ascending=False)

top_features_all = final_rank.head(50)
print("\n--- Top Features (Audit) ---")
print(top_features_all.head(10))

magic_25 = final_rank.head(25).index.tolist()
print("\nFINAL MAGIC 25 LIST:")
print(magic_25)

import json
with open('magic_25_features_volatility.json', 'w') as f:
    json.dump(magic_25, f)

Iteration 1/3...
Dropping 0 constant columns.
[LightGBM] [Info] Number of positive: 100000, number of negative: 100000
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 142263
[LightGBM] [Info] Number of data points in the train set: 200000, number of used features: 751
[LightGBM] [Info] Using GPU Device: Intel(R) UHD Graphics, Vendor: Intel(R) Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 403 dense feature groups (77.06 MB) transferred to GPU in 0.060239 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Iteration 2/3...
[LightGBM] [Info] Number of positive: 100000, number of negative: 100000
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 142303
[LightGBM] [Info] Number of data points in the train set: 200000, number of used features: 752
[Lig

By including Standard Deviation in our feature pool, we found that the model prioritized temporal volatility ($B\_3\_std$) over simple averages. This confirms that credit risk is not static; it is a sequence-dependent problem. 


In [24]:
slope_exprs = [
    (
        (pl.col(c).last() - pl.col(c).get(pl.col(c).len() - 7)) / 6
    ).alias(f"{c}_slope")
    for c in ["P_2", "B_1", "B_3", "D_39", "S_3"]
]

slope_df = (
    cleaned_data.lazy()
    .group_by("customer_ID")
    .agg([
        pl.count().alias("row_count"),
        *slope_exprs
    ])
    .filter(pl.col("row_count") >= 7)  
    .collect()
)

C:\Users\HP\AppData\Local\Temp\ipykernel_1748\3448396050.py:12: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("row_count"),


We calculated the Linear Slope of the raw 13-month sequences for our top features. We found that even when users shared identical 'Mean' and 'Last' values, the Direction of Change was a decisive factor in predicting default.

In [25]:
final_audit_df = slope_df.join(volatility_df, on="customer_ID")


recovering_cases = final_audit_df.filter(
    (pl.col("P_2_mean") < 0.3) & (pl.col("P_2_slope") > 0.05)
).head(3)


spiraling_cases = final_audit_df.filter(
    (pl.col("P_2_mean") > 0.7) & (pl.col("P_2_slope") < -0.05)
).head(3)

In [26]:
recovering_cases

customer_ID,row_count,P_2_slope,B_1_slope,B_3_slope,D_39_slope,S_3_slope,target,P_2_mean,D_39_mean,B_1_mean,B_2_mean,R_1_mean,S_3_mean,D_41_mean,B_3_mean,D_42_mean,D_43_mean,D_44_mean,B_4_mean,D_45_mean,B_5_mean,R_2_mean,D_46_mean,D_47_mean,D_48_mean,D_49_mean,B_6_mean,B_7_mean,B_8_mean,D_50_mean,D_51_mean,B_9_mean,R_3_mean,D_52_mean,P_3_mean,B_10_mean,…,S_27_delta,D_113_delta,D_114_delta,D_115_delta,D_116_delta,D_117_delta,D_118_delta,D_119_delta,D_120_delta,D_121_delta,D_122_delta,D_123_delta,D_124_delta,D_125_delta,D_126_delta,D_127_delta,D_128_delta,D_129_delta,B_41_delta,B_42_delta,D_130_delta,D_131_delta,D_132_delta,D_133_delta,R_28_delta,D_134_delta,D_135_delta,D_136_delta,D_137_delta,D_138_delta,D_139_delta,D_140_delta,D_141_delta,D_142_delta,D_143_delta,D_144_delta,D_145_delta
str,u32,f32,f32,f32,f64,f32,i64,f32,f64,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f32,f32,f64,f32,f32,f32,f64,f32,f32,f32,f32,f64,f32,f64,f32,f32,f32,…,f32,f64,f64,f32,f64,f64,f32,f32,f64,f32,f64,f64,f64,f64,f64,f64,f32,f64,f64,f32,f32,f32,f32,f32,f64,f32,f64,f64,f64,f64,f64,f64,f32,f32,f64,f32,f64
"""6ec380f80c9c7f59f12b7da1bd692c…",7,166.642105,0.001222,0.000002,0.0,166.519379,0,-141.980057,4.857143,0.081477,0.807902,0.004406,-142.605423,0.0,0.005401,-142.709457,-285.399963,0.0,1.571429,0.075595,0.032401,0.0,-856.219055,0.593582,0.015015,-1.0,0.358884,0.081307,0.859341,-999.0,1.571429,0.075638,1.0,0.248767,-570.593079,0.259014,…,-142.846115,-0.285714,-0.571429,-285.58902,-0.285714,-0.285714,-285.61145,-285.619446,-0.714286,-285.594025,-0.571429,-0.285714,-2.285714,-0.285714,-0.857143,0.0,-285.428558,-0.285714,0.0,0.0,-285.428558,-285.428558,0.0,-285.428894,0.0,0.0,0.0,0.0,0.0,0.0,-0.285714,-0.285714,-285.428558,0.0,-0.285714,-285.431458,-0.285714
"""3e2e44820ab5252737584af522d46c…",8,0.091488,-0.219775,0.000962,-1.5,-0.052921,0,-374.502838,1.25,0.268779,0.863676,0.004737,0.283898,0.0,0.004015,2.42424,-999.0,-1.0,0.875,0.015627,0.112665,0.0,-999.0,0.014413,-873.966553,-1.0,0.211903,0.253471,1.002993,-999.0,0.0,0.114843,1.375,1.005173,-999.0,0.256431,…,624.653076,-1.0,-0.5,-499.504639,-0.5,-0.5,-499.510315,-499.504211,-1.0,-499.49588,-1.0,-0.5,-1.0,-1.0,-1.0,0.0,-249.75,-0.25,0.0,0.0,-249.75,-249.75,0.0,-124.874023,0.0,0.0,0.0,0.0,0.0,0.0,-0.25,-0.125,-249.75,0.0,-0.25,-124.87384,-0.25
"""6195e918fe50a5b41ed0782ca6e986…",7,166.643707,0.003522,-0.000392,0.0,166.51683,0,-141.975861,4.0,0.060589,0.92477,0.007156,-142.581848,0.0,0.004861,-142.707489,0.048165,0.0,3.857143,0.012219,0.053924,0.0,0.419299,0.442353,0.041026,-1.0,0.309412,0.049083,1.006503,-999.0,0.0,0.081686,1.0,0.197138,0.619948,0.265246,…,-571.268127,-0.428571,-0.428571,-428.300507,-0.428571,1.285714,-428.241943,-428.241974,-0.857143,-428.449249,-2.142857,-0.428571,-7.285714,-0.428571,-1.0,0.0,-285.428558,-0.285714,0.0,0.0,-285.428558,-285.428558,0.0,-142.718399,0.0,0.0,0.0,0.0,0.0,0.0,-0.285714,-0.142857,-285.428558,0.0,-0.285714,-142.711456,-0.285714


In [27]:
spiraling_cases 

customer_ID,row_count,P_2_slope,B_1_slope,B_3_slope,D_39_slope,S_3_slope,target,P_2_mean,D_39_mean,B_1_mean,B_2_mean,R_1_mean,S_3_mean,D_41_mean,B_3_mean,D_42_mean,D_43_mean,D_44_mean,B_4_mean,D_45_mean,B_5_mean,R_2_mean,D_46_mean,D_47_mean,D_48_mean,D_49_mean,B_6_mean,B_7_mean,B_8_mean,D_50_mean,D_51_mean,B_9_mean,R_3_mean,D_52_mean,P_3_mean,B_10_mean,…,S_27_delta,D_113_delta,D_114_delta,D_115_delta,D_116_delta,D_117_delta,D_118_delta,D_119_delta,D_120_delta,D_121_delta,D_122_delta,D_123_delta,D_124_delta,D_125_delta,D_126_delta,D_127_delta,D_128_delta,D_129_delta,B_41_delta,B_42_delta,D_130_delta,D_131_delta,D_132_delta,D_133_delta,R_28_delta,D_134_delta,D_135_delta,D_136_delta,D_137_delta,D_138_delta,D_139_delta,D_140_delta,D_141_delta,D_142_delta,D_143_delta,D_144_delta,D_145_delta
str,u32,f32,f32,f32,f64,f32,i64,f32,f64,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f32,f32,f64,f32,f32,f32,f64,f32,f32,f32,f32,f64,f32,f64,f32,f32,f32,…,f32,f64,f64,f32,f64,f64,f32,f32,f64,f32,f64,f64,f64,f64,f64,f64,f32,f64,f64,f32,f32,f32,f32,f32,f64,f32,f64,f64,f64,f64,f64,f64,f32,f32,f64,f32,f64
"""6e2813eaec3589154d2764dba675b7…",13,-0.064756,-0.000422,0.000389,0.0,0.0,0,0.757189,0.0,0.005243,0.830035,0.159931,-999.0,0.0,0.00447,-999.0,0.113252,0.0,4.076923,0.204036,0.00706,0.076923,0.439411,0.435065,0.261585,-1.0,0.04864,0.117227,0.0,-999.0,0.307692,0.004471,0.615385,0.143406,0.488715,0.054478,…,0.0,0.0,0.0,-0.008235,0.0,0.0,-0.011907,-0.014267,0.0,-0.01375,0.0,0.0,0.461538,0.0,0.0,0.0,0.005097,0.0,0.0,0.0,0.0,0.0,0.0,0.002528,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002085,0.0
"""6e7cc6944327a9eb1e94344ecf9da3…",13,-0.051789,-0.0047,-0.056717,-1.0,-0.009081,0,0.832434,8.076923,0.048278,0.866265,0.099751,0.04585,0.0,0.169169,-999.0,-76.835938,0.0,6.230769,0.377903,0.513323,0.0,0.436727,1.148996,-153.674133,-1.0,0.168284,0.045014,0.0,-614.725525,0.769231,0.004801,0.0,0.426535,0.482785,0.459567,…,-0.08208,-0.692308,0.0,0.011238,0.0,-1.153846,0.007384,0.007395,0.0,-0.067552,0.230769,-0.692308,-0.461538,-0.692308,0.0,0.0,0.000699,0.461538,0.0,0.0,0.0,0.0,0.0,-0.002226,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.000162,0.0
"""b1bc19233ac6790ec4db015f6a5f21…",13,-0.109583,-0.00794,0.00239,4.333333,0.006894,1,0.713424,22.0,0.056926,0.805164,0.178816,0.13478,0.188182,0.015078,-999.0,-999.0,0.0,4.230769,0.449388,0.093544,0.153846,0.529065,0.946504,0.094484,-1.0,0.128454,0.071355,0.0,-999.0,0.923077,0.006127,0.0,0.262531,0.153224,0.166855,…,-0.157916,0.0,0.0,-0.009174,0.0,0.230769,-0.015383,-0.008018,0.0,-0.014866,0.0,0.0,0.0,0.0,-0.153846,-0.384615,-0.000021,0.0,0.0,0.0,0.0,0.0,0.0,-0.004809,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.002884,0.0


# Bold Why we calculated slope ?

We calcualted slope to understand the trend of the data over time.Because sometimes the data might be increasing or decreasing over time and we want to capture that trend.For Example: If you are standing at the edge of a cliff, are you walking toward it or away from it? The "Last" value only tells us where you are standing; the Slope tells us which way you are moving.


# Bold Why Temporal Analysis?

Consider Two users : 
1. User A: Had high debt for 10 months but got a high-paying job 3 months ago and has paid off everything since.The model wrongly rejects their loan because it's stuck in the past.
2. User B: Was a perfect "A+" customer for 11 months, but in the last 60 days, their spending has tripled and they stopped paying.The model might fail to highlight this recent change.

User A can be classified as recovered user.
User B can be classified as spiraling user.

We were able to identify these two types of users by calculating the slope of the data over time.

